# Experiment Results — Report Visuals

Polished figures from all experiments and holdout evaluation.

| Section | Experiment | Key question |
|---|---|---|
| 1 | Holdout main | How much does each model improve over popularity? |
| 2 | Holdout distributions | How consistent are gains across users? |
| 3 | Holdout coverage | What do raw metrics hide? |
| 4 | Holdout trade-off | Diversity vs relevance per user |
| 5 | Holdout user-level | Who benefits from persona reranking? |
| 6 | Exp 1 — Weight sensitivity | Which scorer components matter most? |
| 7 | Exp 2 — Persona count | Does BIC-selected k improve recommendations? |
| 8 | Exp 3 — Rec quality | Artist diversity and persona coverage in practice |

In [2]:
import warnings; warnings.filterwarnings('ignore')
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.ticker import FuncFormatter
from pathlib import Path
from itertools import combinations

RES       = Path('../results')
OUT_DIR   = Path('../results/report_figures')
OUT_DIR.mkdir(parents=True, exist_ok=True)

# ── Colour scheme (used throughout) ──────────────────────────────────────────
MODEL_COLOR = {
    'popularity':       '#C44E52',
    'als':              '#4C72B0',
    'persona_reranker': '#55A868',
}
MODEL_LABEL = {
    'popularity':       'Popularity',
    'als':              'ALS',
    'persona_reranker': 'Persona reranker',
}
COMP_COLOR = {
    'sonic_fit':             '#4C72B0',
    'novelty':               '#DD8452',
    'persona_specificity':   '#55A868',
    'familiarity':           '#C44E52',
    'default_weights':       '#8172B2',
}
MODEL_ORDER = ['popularity', 'als', 'persona_reranker']

plt.rcParams.update({
    'figure.dpi': 140,
    'font.size': 11,
    'axes.spines.top': False,
    'axes.spines.right': False,
    'axes.labelsize': 11,
    'axes.titlesize': 12,
    'xtick.labelsize': 10,
    'ytick.labelsize': 10,
})

print('Setup done. Results available in:', [d.name for d in RES.iterdir() if d.is_dir()])

Setup done. Results available in: ['report_figures', 'holdout_top500', 'exp2_best', 'exp3_best', 'persona_report', 'holdout_top250', 'holdout_persona_blend', 'exp1_best']


In [3]:
# ── Load all results ──────────────────────────────────────────────────────────
HOLDOUT_DIR = RES / 'holdout_top500'

summary  = pd.read_csv(HOLDOUT_DIR / 'holdout_summary.csv', index_col=0)
metrics  = pd.read_csv(HOLDOUT_DIR / 'holdout_metrics.csv')

exp1_df  = pd.read_csv(RES / 'exp1_best/weight_sensitivity_results.csv')
exp2_df  = pd.read_csv(RES / 'exp2_best/persona_validation_results.csv')
bic_df   = pd.read_csv(RES / 'exp2_best/bic_curves.csv')
exp3_df  = pd.read_csv(RES / 'exp3_best/recommendation_quality.csv')

print('Holdout users :', metrics['user_id'].nunique())
print('Holdout models:', metrics['model'].unique().tolist())
print('Summary:\n', summary[['Recall@10','Recall_adj@10','NDCG@10','NDCG_adj@10','HitRate@10','ILD']].round(4).to_string())

Holdout users : 1212
Holdout models: ['popularity', 'als', 'persona_reranker']
Summary:
                   Recall@10  Recall_adj@10  NDCG@10  NDCG_adj@10  HitRate@10     ILD
model                                                                               
als                  0.0359         0.0477   0.0408       0.0484      0.2434  0.7346
persona_reranker     0.0321         0.0417   0.0372       0.0438      0.2203  0.7635
popularity           0.0161         0.0206   0.0199       0.0230      0.1254  0.8760


## 1. Holdout — Main Results

Mean metrics on the test split across all evaluation users.

In [4]:
available = [m for m in MODEL_ORDER if m in summary.index]
k = 10

metric_groups = [
    # (raw_col, adj_col, title, show_adj)
    (f'Recall@{k}',   f'Recall_adj@{k}',  f'Recall@{k}',   True),
    (f'NDCG@{k}',     f'NDCG_adj@{k}',    f'NDCG@{k}',     True),
    (f'HitRate@{k}',  None,               f'Hit Rate@{k}',  False),
    ('ILD',           None,               'Intra-List\nDiversity', False),
    ('long_tail_pct', None,               'Long-tail\nShare',      False),
]

fig, axes = plt.subplots(1, len(metric_groups), figsize=(18, 5))
x = np.arange(len(available))
w = 0.38

for ax, (raw_col, adj_col, title, show_adj) in zip(axes, metric_groups):
    raw_vals = [summary.loc[m, raw_col] if raw_col in summary.columns else 0 for m in available]

    if show_adj and adj_col and adj_col in summary.columns:
        adj_vals = [summary.loc[m, adj_col] for m in available]
        bars_r = ax.bar(x - w/2, raw_vals, w,
                        color=[MODEL_COLOR[m] for m in available],
                        alpha=0.45, label='Raw')
        bars_a = ax.bar(x + w/2, adj_vals, w,
                        color=[MODEL_COLOR[m] for m in available],
                        alpha=0.92, label='Coverage-adj.')
        for i, (rv, av) in enumerate(zip(raw_vals, adj_vals)):
            ax.text(i - w/2, rv + 0.001, f'{rv:.3f}', ha='center', va='bottom', fontsize=7.5)
            ax.text(i + w/2, av + 0.001, f'{av:.3f}', ha='center', va='bottom', fontsize=7.5)
        if ax == axes[0]:
            ax.legend(fontsize=8, loc='upper left')
    else:
        bars = ax.bar(x, raw_vals, 0.55,
                      color=[MODEL_COLOR[m] for m in available], alpha=0.88)
        for i, v in enumerate(raw_vals):
            ax.text(i, v + 0.002, f'{v:.3f}', ha='center', va='bottom', fontsize=8)

    ax.set_xticks(x)
    ax.set_xticklabels([MODEL_LABEL[m] for m in available], rotation=20, ha='right')
    ax.set_title(title)
    ax.set_ylim(0, max(raw_vals) * 1.30 + 0.01)
    ax.grid(axis='y', alpha=0.2)

legend_patches = [mpatches.Patch(color=MODEL_COLOR[m], label=MODEL_LABEL[m]) for m in available]
fig.legend(handles=legend_patches, loc='upper center', ncol=3,
           bbox_to_anchor=(0.5, 1.02), fontsize=10, frameon=False)
fig.suptitle('Holdout evaluation — test split  (val used for weight optimisation only)',
             fontsize=13, fontweight='bold', y=1.06)
fig.tight_layout()
fig.savefig(OUT_DIR / '1_holdout_main.png', bbox_inches='tight', dpi=150)
plt.show()

# Lift over popularity
pop_recall = summary.loc['popularity', f'Recall_adj@{k}']
print('Lift over popularity (Recall_adj@10):')
for m in ['als', 'persona_reranker']:
    v = summary.loc[m, f'Recall_adj@{k}']
    print(f'  {MODEL_LABEL[m]:20s}: +{100*(v/pop_recall - 1):.1f}%')

Lift over popularity (Recall_adj@10):
  ALS                 : +131.1%
  Persona reranker    : +101.8%


## 2. Holdout — Per-User Distributions

Means can hide skew. Violin plots show the full distribution of Recall, NDCG, and ILD across users.

In [5]:
plot_cols = [
    ('Recall_adj@10', f'Recall@{k} (coverage-adj.)'),
    ('NDCG_adj@10',   f'NDCG@{k} (coverage-adj.)'),
    ('ILD',           'Intra-List Diversity'),
]

fig, axes = plt.subplots(1, len(plot_cols), figsize=(15, 5))

for ax, (col, ylabel) in zip(axes, plot_cols):
    data_by_model = [
        metrics[metrics['model'] == m][col].dropna().values
        for m in available
    ]
    parts = ax.violinplot(data_by_model, positions=range(len(available)),
                          showmedians=True, showextrema=False)
    for pc, m in zip(parts['bodies'], available):
        pc.set_facecolor(MODEL_COLOR[m])
        pc.set_alpha(0.65)
    parts['cmedians'].set_color('white')
    parts['cmedians'].set_linewidth(2)

    # Overlay jittered points
    rng = np.random.default_rng(42)
    for i, (vals, m) in enumerate(zip(data_by_model, available)):
        jitter = rng.uniform(-0.08, 0.08, len(vals))
        ax.scatter(i + jitter, vals, s=4, alpha=0.25,
                   color=MODEL_COLOR[m], rasterized=True)
        ax.scatter(i, np.mean(vals), s=60, marker='D',
                   color=MODEL_COLOR[m], edgecolors='white', linewidths=1.2, zorder=5)

    ax.set_xticks(range(len(available)))
    ax.set_xticklabels([MODEL_LABEL[m] for m in available], rotation=15, ha='right')
    ax.set_ylabel(ylabel)
    ax.set_title(ylabel)
    ax.grid(axis='y', alpha=0.2)

fig.suptitle('Per-user metric distributions  (◆ = mean, white line = median)',
             fontsize=13, fontweight='bold')
fig.tight_layout()
fig.savefig(OUT_DIR / '2_holdout_distributions.png', bbox_inches='tight', dpi=150)
plt.show()

## 3. Holdout — The Coverage Gap

~33% of held-out songs are not in the ALS embedding catalog. Raw metrics penalise the model for a catalog gap, not ranking failures. Coverage-adjusted metrics reveal the true ranking quality.

In [6]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# Panel 1: catalog coverage bar
ax = axes[0]
coverage_val  = summary['val_coverage'].mean()  if 'val_coverage'  in summary.columns else None
coverage_test = summary['test_coverage'].mean() if 'test_coverage' in summary.columns else None

if coverage_test is not None:
    categories = ['Embeddable\n(in catalog)', 'Not embeddable\n(no ALS vector)']
    vals_test  = [coverage_test, 1 - coverage_test]
    bars = ax.bar(categories, [v * 100 for v in vals_test],
                  color=['#4C72B0', '#dddddd'], alpha=0.9, edgecolor='white')
    ax.set_ylabel('% of held-out test songs')
    ax.set_title(f'Test-set Catalog Coverage\n({coverage_test*100:.1f}% of songs have embeddings)')
    ax.set_ylim(0, 110)
    for bar, v in zip(bars, [v * 100 for v in vals_test]):
        ax.text(bar.get_x() + bar.get_width()/2, v + 1,
                f'{v:.1f}%', ha='center', va='bottom', fontsize=11, fontweight='bold')
    ax.grid(axis='y', alpha=0.2)

# Panel 2: raw vs adjusted side by side (Recall)
ax = axes[1]
for metric_pair, offset, alpha, lbl in [
    ('Recall@10', -0.22, 0.45, 'Raw'),
    ('Recall_adj@10', +0.22, 0.92, 'Coverage-adj.'),
]:
    vals = [summary.loc[m, metric_pair] for m in available]
    ax.bar(np.arange(len(available)) + offset, vals, 0.40,
           color=[MODEL_COLOR[m] for m in available], alpha=alpha, label=lbl)
ax.set_xticks(range(len(available)))
ax.set_xticklabels([MODEL_LABEL[m] for m in available], rotation=20, ha='right')
ax.set_ylabel('Recall@10')
ax.set_title('Raw vs Coverage-adjusted Recall@10')
ax.legend(fontsize=9)
ax.grid(axis='y', alpha=0.2)

# Panel 3: per-user test coverage histogram
ax = axes[2]
if 'test_coverage' in metrics.columns:
    user_cov = metrics[metrics['model'] == 'als']['test_coverage']
    ax.hist(user_cov, bins=30, color='#4C72B0', alpha=0.8, edgecolor='none')
    ax.axvline(user_cov.mean(), color='red', lw=2, ls='--',
               label=f'Mean = {user_cov.mean():.3f}')
    ax.set_xlabel('Fraction of test songs with embeddings')
    ax.set_ylabel('Users')
    ax.set_title('Per-user Test Coverage Distribution')
    ax.legend(fontsize=9)
    ax.grid(alpha=0.2)

fig.suptitle('Catalog coverage gap: ~33% of held-out songs have no ALS embedding',
             fontsize=13, fontweight='bold')
fig.tight_layout()
fig.savefig(OUT_DIR / '3_holdout_coverage.png', bbox_inches='tight', dpi=150)
plt.show()
print(f'Test-set catalog coverage: {coverage_test*100:.1f}%')
print(f'Raw → adjusted Recall lift (ALS): +{100*(summary.loc["als","Recall_adj@10"]/summary.loc["als","Recall@10"]-1):.1f}%')

Test-set catalog coverage: 67.3%
Raw → adjusted Recall lift (ALS): +32.9%


## 4. Holdout — Diversity–Relevance Trade-off

Each dot = one user. Best model: top-right (high diversity AND high relevance).

In [7]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5.5))

# Scatter ILD vs Recall_adj per user
ax = axes[0]
for m in available:
    sub = metrics[metrics['model'] == m]
    x   = sub['ILD'].to_numpy()
    y   = sub['Recall_adj@10'].to_numpy()
    ax.scatter(x, y, s=8, alpha=0.30, color=MODEL_COLOR[m], rasterized=True)
    ax.scatter(x.mean(), y.mean(), s=150, marker='D',
               color=MODEL_COLOR[m], edgecolors='white', linewidths=1.5,
               zorder=6, label=f"{MODEL_LABEL[m]}  ({x.mean():.3f}, {y.mean():.3f})")
ax.set_xlabel('Intra-List Diversity (ILD)')
ax.set_ylabel('Recall@10 (coverage-adj.)')
ax.set_title('Diversity – Relevance per User\n(◆ = model mean)')
ax.legend(fontsize=8.5, frameon=False)
ax.grid(alpha=0.18)

# ILD vs mean_popularity_rank
ax = axes[1]
for m in available:
    sub = metrics[metrics['model'] == m]
    x   = sub['ILD'].to_numpy()
    y   = np.log1p(sub['mean_popularity_rank'].to_numpy())
    ax.scatter(x, y, s=8, alpha=0.30, color=MODEL_COLOR[m], rasterized=True)
    ax.scatter(x.mean(), y.mean(), s=150, marker='D',
               color=MODEL_COLOR[m], edgecolors='white', linewidths=1.5,
               zorder=6, label=MODEL_LABEL[m])
ax.set_xlabel('Intra-List Diversity (ILD)')
ax.set_ylabel('log(Mean popularity rank + 1)')
ax.set_title('Diversity – Popularity Rank\n(higher rank = less popular = more novel)')
ax.legend(fontsize=8.5, frameon=False)
ax.grid(alpha=0.18)

fig.suptitle('Trade-off analysis: persona reranker shifts toward diversity without collapsing relevance',
             fontsize=12, fontweight='bold')
fig.tight_layout()
fig.savefig(OUT_DIR / '4_holdout_tradeoff.png', bbox_inches='tight', dpi=150)
plt.show()

## 5. Holdout — Who Benefits from Persona Reranking?

Per-user difference: persona_reranker recall − ALS recall.
Positive = persona reranker helped; negative = raw ALS was better for this user.

In [8]:
als_u    = metrics[metrics['model'] == 'als'].set_index('user_id')['Recall_adj@10']
persona_u = metrics[metrics['model'] == 'persona_reranker'].set_index('user_id')['Recall_adj@10']
common   = als_u.index.intersection(persona_u.index)
delta    = (persona_u[common] - als_u[common]).sort_values()

fig, axes = plt.subplots(1, 3, figsize=(17, 5))

# Delta histogram
ax = axes[0]
neg = (delta < 0).sum(); pos = (delta > 0).sum(); zero = (delta == 0).sum()
ax.hist(delta[delta < 0],  bins=40, color='#C44E52', alpha=0.8, label=f'ALS better  (n={neg})')
ax.hist(delta[delta == 0], bins=1,  color='#aaaaaa', alpha=0.8, label=f'Tied  (n={zero})')
ax.hist(delta[delta > 0],  bins=40, color='#55A868', alpha=0.8, label=f'Persona better  (n={pos})')
ax.axvline(0, color='black', lw=1, ls='--')
ax.axvline(delta.mean(), color='orange', lw=1.5, ls='--',
           label=f'Mean Δ = {delta.mean():.4f}')
ax.set_xlabel('Persona Recall@10 − ALS Recall@10')
ax.set_ylabel('Users')
ax.set_title('Per-user Recall Difference\n(persona vs ALS)')
ax.legend(fontsize=8)
ax.grid(alpha=0.2)

# Sorted delta (waterfall)
ax = axes[1]
colors_d = ['#55A868' if v >= 0 else '#C44E52' for v in delta.values]
ax.bar(range(len(delta)), delta.values, color=colors_d, alpha=0.7, width=1.0)
ax.axhline(0, color='black', lw=0.8)
ax.set_xlabel('Users (sorted by Δ recall)')
ax.set_ylabel('Δ Recall@10')
ax.set_title('Sorted per-user Δ Recall\n(persona reranker − ALS)')
ax.grid(axis='y', alpha=0.2)

# ALS recall vs Persona recall scatter
ax = axes[2]
ax.scatter(als_u[common], persona_u[common], s=8, alpha=0.35,
           color='#8172B2', rasterized=True)
lim = max(als_u[common].max(), persona_u[common].max()) * 1.05
ax.plot([0, lim], [0, lim], 'k--', lw=1, alpha=0.4, label='Equal performance')
ax.set_xlabel('ALS Recall@10')
ax.set_ylabel('Persona Reranker Recall@10')
ax.set_title('ALS vs Persona Recall per User\n(above diagonal = persona wins)')
ax.legend(fontsize=9)
ax.grid(alpha=0.18)

fig.suptitle(f'Persona reranker helps {pos} users, hurts {neg}, ties {zero}  (out of {len(common)} total)',
             fontsize=12, fontweight='bold')
fig.tight_layout()
fig.savefig(OUT_DIR / '5_holdout_user_benefit.png', bbox_inches='tight', dpi=150)
plt.show()

## 6. Exp 1 — Scoring Weight Sensitivity

Grid search over all pairs of scorer component weights. Shows which components drive NDCG@10 and validates the default weight choice.

In [9]:
COMP_NAMES   = ['sonic_fit', 'novelty', 'persona_specificity', 'familiarity']
COMP_DISPLAY = {
    'sonic_fit':           'Sonic fit',
    'novelty':             'Novelty',
    'persona_specificity': 'Persona\nspecificity',
    'familiarity':         'Familiarity',
}

grid = exp1_df[exp1_df['analysis'] == 'pairwise_heatmap'].copy()
pairs = list(combinations(COMP_NAMES, 2))

fig, axes = plt.subplots(2, 3, figsize=(18, 11))
axes = axes.flatten()
has_map = False

for ax, (x_name, y_name) in zip(axes, pairs):
    pair_df = grid[(grid['sweep_x'] == x_name) & (grid['sweep_y'] == y_name)]
    if pair_df.empty:
        ax.set_axis_off()
        continue
    wx_col = f'w_{x_name}'
    wy_col = f'w_{y_name}'
    if wx_col not in pair_df.columns or wy_col not in pair_df.columns:
        ax.set_axis_off()
        continue

    heat = (
        pair_df.groupby([wy_col, wx_col])['ndcg'].mean()
        .reset_index()
        .pivot(index=wy_col, columns=wx_col, values='ndcg')
    )
    if heat.empty:
        ax.set_axis_off()
        continue

    data = heat.to_numpy(dtype=float)
    vmax = np.nanpercentile(data, 98)
    im = ax.imshow(data, cmap='YlOrRd', vmin=0, vmax=vmax,
                   origin='lower', aspect='auto')
    plt.colorbar(im, ax=ax, shrink=0.85, label='NDCG@10')

    # Mark best cell
    best_r, best_c = np.unravel_index(np.nanargmax(data), data.shape)
    ax.scatter(best_c, best_r, s=120, marker='*', color='white',
               zorder=5, label=f'Best: {np.nanmax(data):.3f}')
    ax.legend(fontsize=8, loc='lower right', frameon=False)

    ax.set_xticks(np.arange(len(heat.columns)))
    ax.set_yticks(np.arange(len(heat.index)))
    step = max(1, len(heat.columns) // 6)
    ax.set_xticklabels(
        [f'{v:.1f}' if i % step == 0 else '' for i, v in enumerate(heat.columns)],
        rotation=45, ha='right', fontsize=8)
    ax.set_yticklabels(
        [f'{v:.1f}' if i % step == 0 else '' for i, v in enumerate(heat.index)],
        fontsize=8)
    ax.set_xlabel(f'Weight: {COMP_DISPLAY[x_name]}', fontsize=9)
    ax.set_ylabel(f'Weight: {COMP_DISPLAY[y_name]}', fontsize=9)
    ax.set_title(f'{COMP_DISPLAY[x_name]} vs {COMP_DISPLAY[y_name]}', fontsize=10)
    has_map = True

if not has_map:
    print('No pairwise heatmap data found — check exp1 CSV.')
else:
    fig.suptitle('Scoring weight sensitivity — mean NDCG@10 across all pairwise sweeps\n'
                 '(★ = best cell per pair; other weights rescaled from default)',
                 fontsize=13, fontweight='bold')
    fig.tight_layout()
    fig.savefig(OUT_DIR / '6_exp1_heatmaps.png', bbox_inches='tight', dpi=150)
    plt.show()

In [10]:
ablation_df = exp1_df[exp1_df['analysis'] == 'ablation'].copy()
ABLATION_ORDER = [
    'sonic_fit_only', 'novelty_only',
    'persona_specificity_only', 'familiarity_only', 'default_weights',
]
ABLATION_LABEL = {
    'sonic_fit_only':           'Sonic fit\nonly',
    'novelty_only':             'Novelty\nonly',
    'persona_specificity_only': 'Persona\nspec. only',
    'familiarity_only':         'Familiarity\nonly',
    'default_weights':          'Default\n(combined)',
}
avail_labels = [l for l in ABLATION_ORDER if l in ablation_df['label'].values]

abl_mean = ablation_df.groupby('label')['ndcg'].mean().reindex(avail_labels)
abl_std  = ablation_df.groupby('label')['ndcg'].std().reindex(avail_labels)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Ablation bar with error
ax = axes[0]
bar_colors = [COMP_COLOR.get(l.replace('_only',''), '#8172B2') for l in avail_labels]
bars = ax.bar(range(len(avail_labels)), abl_mean.values, 0.60,
              yerr=abl_std.values, capsize=4,
              color=bar_colors, alpha=0.88, edgecolor='white',
              error_kw={'elinewidth': 1.5, 'ecolor': 'black', 'alpha': 0.6})
ax.set_xticks(range(len(avail_labels)))
ax.set_xticklabels([ABLATION_LABEL[l] for l in avail_labels], rotation=20, ha='right')
ax.set_ylabel('Mean NDCG@10')
ax.set_title('Component Ablation\n(mean ± 1 std across users)')
for i, (v, s) in enumerate(zip(abl_mean.values, abl_std.values)):
    ax.text(i, v + s + 0.002, f'{v:.4f}', ha='center', fontsize=8.5)
ax.grid(axis='y', alpha=0.2)

# Per-user distribution for each ablation config
ax = axes[1]
for i, label in enumerate(avail_labels):
    vals = ablation_df[ablation_df['label'] == label]['ndcg'].values
    parts = ax.violinplot([vals], positions=[i], showmedians=True, showextrema=False)
    for pc in parts['bodies']:
        pc.set_facecolor(bar_colors[i])
        pc.set_alpha(0.65)
    parts['cmedians'].set_color('white')
    parts['cmedians'].set_linewidth(2)
ax.set_xticks(range(len(avail_labels)))
ax.set_xticklabels([ABLATION_LABEL[l] for l in avail_labels], rotation=20, ha='right')
ax.set_ylabel('NDCG@10')
ax.set_title('Ablation Distributions per User')
ax.grid(axis='y', alpha=0.2)

fig.suptitle('Exp 1 — Which scorer component contributes most to ranking quality?',
             fontsize=13, fontweight='bold')
fig.tight_layout()
fig.savefig(OUT_DIR / '7_exp1_ablation.png', bbox_inches='tight', dpi=150)
plt.show()
print('Mean NDCG by component:')
for l in avail_labels:
    print(f'  {ABLATION_LABEL[l].replace(chr(10)," "):25s}: {abl_mean[l]:.4f}')

Mean NDCG by component:
  Sonic fit only           : 0.0240
  Novelty only             : 0.0000
  Persona spec. only       : 0.0000
  Familiarity only         : 0.0012
  Default (combined)       : 0.0122


## 7. Exp 2 — Persona Count Validation

Does BIC-selected k improve ranking quality? Does user type (niche/mainstream/diverse) drive the optimal k?

In [ ]:
TYPE_COLOR = {'niche': '#4C72B0', 'mainstream': '#DD8452', 'diverse': '#55A868'}
TYPE_ORDER = ['niche', 'mainstream', 'diverse']

fig, axes = plt.subplots(1, 3, figsize=(17, 5.5))

# NDCG vs k by user type (with individual user lines)
ax = axes[0]
for utype in TYPE_ORDER:
    sub = exp2_df[exp2_df['user_type'] == utype]
    if sub.empty:
        continue
    mean_k  = sub.groupby('k')['ndcg'].mean()
    std_k   = sub.groupby('k')['ndcg'].std().fillna(0)
    ks_u    = mean_k.index.to_numpy()
    ax.fill_between(ks_u, mean_k - std_k, mean_k + std_k,
                    alpha=0.15, color=TYPE_COLOR[utype])
    ax.plot(ks_u, mean_k.values, 'o-', lw=2, ms=6,
            color=TYPE_COLOR[utype], label=utype.title())
    # Star at BIC mode k
    bic_ks = sub[sub['is_bic']]['k']
    if not bic_ks.empty:
        mode_k = bic_ks.mode().iloc[0]
        if mode_k in mean_k.index:
            ax.scatter([mode_k], [mean_k[mode_k]], s=250, marker='*',
                       color=TYPE_COLOR[utype], zorder=5)

ax.set_xlabel('k (number of personas)')
ax.set_ylabel('NDCG@10')
ax.set_title('NDCG@10 vs Persona Count\nby User Type  (★ = BIC-selected k)')
ax.legend(fontsize=9)
ax.grid(alpha=0.2)

# BIC curves — mean normalized BIC by user type
ax = axes[1]
for utype in TYPE_ORDER:
    sub_bic = bic_df[bic_df['user_type'] == utype]
    if sub_bic.empty:
        continue
    # Normalize BIC per user so they're comparable
    norm_bic = sub_bic.groupby('user_id')['bic'].transform(lambda x: (x - x.min()) / (x.max() - x.min() + 1e-9))
    sub_bic = sub_bic.copy()
    sub_bic['norm_bic'] = norm_bic
    mean_bic = sub_bic.groupby('k')['norm_bic'].mean()
    ax.plot(mean_bic.index, mean_bic.values, 'o-', lw=2, ms=6,
            color=TYPE_COLOR[utype], label=utype.title())

ax.set_xlabel('k')
ax.set_ylabel('Normalized BIC  (0 = min, 1 = max per user)')
ax.set_title('BIC Score vs k by User Type\n(lower = better fit; where does BIC bottom out?)')
ax.legend(fontsize=9)
ax.grid(alpha=0.2)

# BIC-selected k distribution by user type
ax = axes[2]
bic_k_by_type = exp2_df.drop_duplicates('user_id').groupby('user_type')['bic_k']
type_present  = [t for t in TYPE_ORDER if t in bic_df['user_type'].unique()]
for i, utype in enumerate(type_present):
    sub_exp = exp2_df[exp2_df['user_type'] == utype].drop_duplicates('user_id')
    ks_u    = sub_exp['bic_k'].values
    jitter  = np.random.default_rng(42).uniform(-0.15, 0.15, len(ks_u))
    ax.scatter(i + jitter, ks_u, s=30, alpha=0.6,
               color=TYPE_COLOR[utype], rasterized=True)
    ax.scatter(i, np.mean(ks_u), s=150, marker='D',
               color=TYPE_COLOR[utype], edgecolors='white', linewidths=1.5, zorder=5)

ax.set_xticks(range(len(type_present)))
ax.set_xticklabels([t.title() for t in type_present])
ax.set_ylabel('BIC-selected k')
ax.set_title('BIC-selected k by User Type\n(◆ = mean)')
ax.grid(axis='y', alpha=0.2)

fig.suptitle('Exp 2 — Persona count validation: BIC adapts complexity to user listening pattern',
             fontsize=13, fontweight='bold')
fig.tight_layout()
fig.savefig(OUT_DIR / '8_exp2_persona_validation.png', bbox_inches='tight', dpi=150)
plt.show()

## 8. Exp 3 — Recommendation Quality Dashboard

Discovery rate, NDCG, artist repetition, and persona coverage across 68 users.

In [12]:
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
rng = np.random.default_rng(42)

# 1. Metric distributions (violin)
metric_info = [
    ('discovery_at_k',       'Discovery Rate@10'),
    ('ndcg_at_k',            'NDCG@10'),
    ('artist_repetition_rate','Artist Repetition Rate'),
    ('persona_coverage',     'Persona Coverage'),
]
for idx, (col, label) in enumerate(metric_info):
    ax = axes[idx // 2][idx % 2] if idx < 4 else None
    if ax is None:
        break
    vals = exp3_df[col].dropna().values
    parts = ax.violinplot([vals], positions=[0], showmedians=True, showextrema=False)
    for pc in parts['bodies']:
        pc.set_facecolor('#4C72B0'); pc.set_alpha(0.65)
    parts['cmedians'].set_color('white'); parts['cmedians'].set_linewidth(2)
    jitter = rng.uniform(-0.06, 0.06, len(vals))
    ax.scatter(jitter, vals, s=18, alpha=0.45, color='#4C72B0', rasterized=True)
    ax.scatter(0, np.mean(vals), s=100, marker='D',
               color='#4C72B0', edgecolors='white', linewidths=1.5, zorder=5,
               label=f'Mean={np.mean(vals):.3f}')
    ax.scatter(0, np.median(vals), s=60, marker='o',
               color='red', edgecolors='white', linewidths=1.2, zorder=5,
               label=f'Median={np.median(vals):.3f}')
    ax.set_xticks([])
    ax.set_ylabel(label)
    ax.set_title(label)
    ax.legend(fontsize=9)
    ax.grid(axis='y', alpha=0.2)

# 5. Persona coverage vs n_personas scatter
ax = axes[1][1] if len(axes[1]) > 1 else axes[1][0]
sc = ax.scatter(exp3_df['n_personas'], exp3_df['persona_coverage'],
                s=40, alpha=0.6, c=exp3_df['ndcg_at_k'], cmap='YlOrRd')
plt.colorbar(sc, ax=ax, label='NDCG@10')
bin_edges = np.arange(0.5, exp3_df['n_personas'].max() + 1.5)
binned = exp3_df.groupby('n_personas')['persona_coverage'].mean()
ax.plot(binned.index, binned.values, 'b-o', lw=2, ms=5, label='Bin mean')
ax.set_xlabel('Number of personas (k)')
ax.set_ylabel('Persona coverage')
ax.set_title('Persona Coverage vs k\n(coloured by NDCG)')
ax.legend(fontsize=9)
ax.grid(alpha=0.2)

# 6. Artist repetition vs discovery scatter
ax = axes[1][2] if len(axes[1]) > 2 else axes[1][0]
sc2 = ax.scatter(exp3_df['artist_repetition_rate'], exp3_df['discovery_at_k'],
                 s=40, alpha=0.6, c=exp3_df['persona_coverage'], cmap='cool')
plt.colorbar(sc2, ax=ax, label='Persona coverage')
ax.set_xlabel('Artist Repetition Rate')
ax.set_ylabel('Discovery Rate@10')
ax.set_title('Artist Repetition vs Discovery\n(coloured by persona coverage)')
ax.grid(alpha=0.2)

fig.suptitle('Exp 3 — Recommendation quality metrics across 68 users',
             fontsize=13, fontweight='bold')
fig.tight_layout()
fig.savefig(OUT_DIR / '9_exp3_quality.png', bbox_inches='tight', dpi=150)
plt.show()
print('Summary stats:')
print(exp3_df[['discovery_at_k','ndcg_at_k','artist_repetition_rate','persona_coverage']].describe().round(3).to_string())

Summary stats:
       discovery_at_k  ndcg_at_k  artist_repetition_rate  persona_coverage
count          68.000     68.000                  68.000            68.000
mean            0.009      0.019                   0.174             0.409
std             0.029      0.065                   0.200             0.350
min             0.000      0.000                   0.000             0.125
25%             0.000      0.000                   0.000             0.125
50%             0.000      0.000                   0.200             0.250
75%             0.000      0.000                   0.200             0.688
max             0.100      0.339                   0.800             1.000


## Summary — All Figures

In [13]:
figures = [
    ('1_holdout_main.png',        'Holdout — 5-metric comparison across models'),
    ('2_holdout_distributions.png','Holdout — per-user violin plots'),
    ('3_holdout_coverage.png',    'Holdout — catalog coverage gap (raw vs adj)'),
    ('4_holdout_tradeoff.png',    'Holdout — diversity–relevance trade-off scatter'),
    ('5_holdout_user_benefit.png','Holdout — per-user delta (persona vs ALS)'),
    ('6_exp1_heatmaps.png',       'Exp 1  — 6-pair weight sensitivity heatmaps'),
    ('7_exp1_ablation.png',       'Exp 1  — component ablation bars + violins'),
    ('8_exp2_persona_validation.png','Exp 2  — NDCG vs k + BIC + user type'),
    ('9_exp3_quality.png',        'Exp 3  — recommendation quality dashboard'),
]
print('Saved figures:')
for fname, desc in figures:
    p = OUT_DIR / fname
    print(f"  {'✓' if p.exists() else '○'}  {fname:40s} {desc}")

Saved figures:
  ✓  1_holdout_main.png                       Holdout — 5-metric comparison across models
  ✓  2_holdout_distributions.png              Holdout — per-user violin plots
  ✓  3_holdout_coverage.png                   Holdout — catalog coverage gap (raw vs adj)
  ✓  4_holdout_tradeoff.png                   Holdout — diversity–relevance trade-off scatter
  ✓  5_holdout_user_benefit.png               Holdout — per-user delta (persona vs ALS)
  ✓  6_exp1_heatmaps.png                      Exp 1  — 6-pair weight sensitivity heatmaps
  ✓  7_exp1_ablation.png                      Exp 1  — component ablation bars + violins
  ✓  8_exp2_persona_validation.png            Exp 2  — NDCG vs k + BIC + user type
  ✓  9_exp3_quality.png                       Exp 3  — recommendation quality dashboard
